In [4]:
import os
import sys
import logging
from datetime import datetime  # 1. 导入 datetime 模块
import pandas as pd

from make_ids_map_table_utils import *

In [5]:
# --- 配置 ---
logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)

PROXY_URL = "http://127.0.0.1:7897"
os.environ["http_proxy"] = PROXY_URL
os.environ["https_proxy"] = PROXY_URL

class Config:
    TIMESTAMP = datetime.now().strftime('%y%m%d') 

    IN_DIR = '../data/input'
    IN_OUT_DIR = '../data/in_out'
    OUT_DIR = '../data/output'
    TEMP_DIR = '../tmp_data'

    MISSING_DATA_MAG_DIR = os.path.join(TEMP_DIR, 'mis_data_mag')
    DEDUPLICATE_DIR = os.path.join(TEMP_DIR, 'deduplicate')
    
    # 输入文件路径
    RAW_DATASET = os.path.join(IN_DIR, '0_raw-dataset.csv')
 
    # 输出文件路径
    IDS_MAP_TABLE = os.path.join(IN_OUT_DIR, f'{TIMESTAMP}_raw-ds_fetch-ids_add-ids.csv')
    IDS_MAP_TABLE_OUT = os.path.join(OUT_DIR, f'{TIMESTAMP}_out_ids-map-tb.csv')
    
    BYPRODUCT_TABLE = os.path.join(IN_OUT_DIR, f'{TIMESTAMP}_ids-map-tb_dd_fg.csv')
    BYPRODUCT_TABLE_OUT = os.path.join(OUT_DIR, f'{TIMESTAMP}_out_byproduct-tb.csv')

In [6]:
if not os.path.exists(Config.RAW_DATASET):
    logging.error(f"文件未找到: {Config.RAW_DATASET}")
    sys.exit(1)

logging.info("开始加载数据集...")
file_ext = os.path.splitext(Config.RAW_DATASET)[1].lower()
try:
    if file_ext == '.csv':
        raw_dataset = pd.read_csv(Config.RAW_DATASET, encoding='utf-8-sig') 
    elif file_ext in ['.xlsx', '.xls']:
        raw_dataset = pd.read_excel(Config.RAW_DATASET)
except Exception as e:
    print(f"读取文件出错: {e}")
    sys.exit(1)

compound_array = pd.unique(raw_dataset[['c1', 'c2', 'c3']].values.ravel())
compounds_table = pd.DataFrame(compound_array, columns=['compound']).dropna()
compounds_table = compounds_table[compounds_table['compound'].astype(str).str.strip() != '']

total_compounds = len(compounds_table)
logging.info(f"提取到 {total_compounds} 个需要处理的唯一化合物。")

results_dict = {} # 使用字典存储结果，方便后续根据 name 更新

# ---------------- Phase 1: 初次全量检索 ----------------
logging.info("=== 启动第一轮全量检索 ===")
for idx, name in enumerate(compounds_table['compound'], start=1):
    print(f"\n" + "="*60)
    print(f"📦 初次检索进度: [{idx}/{total_compounds}] | 当前处理: {name}")
    
    percent = idx / total_compounds
    filled_length = int(40 * percent)
    bar = '█' * filled_length + '-' * (40 - filled_length)
    print(f"🚀 总体进度: |{bar}| {percent*100:.1f}%")
    print(f"--> 正在向 PubChem 请求 ...", flush=True)
    
    res = f_fetch_compound_info(name)
    results_dict[name] = res

# ---------------- Phase 2: 审查与二次回炉 ----------------
failed_names = []
for name, res in results_dict.items():
    # 如果 smiles, uname, cas 中有任何一个是 None，就判定为失败/不完整
    if None in res:
        failed_names.append(name)

if failed_names:
    logging.warning(f"\n⚠️ 第一轮检索结束。发现 {len(failed_names)} 个化合物的数据不完整！准备打回重审...")
    # 给服务器一点喘息的时间，防止是因为被封锁导致的集体失败
    cooldown_time = 10 
    logging.info(f"冷静期：休眠 {cooldown_time} 秒后启动第二轮检索...")
    time.sleep(cooldown_time)
    
    # 清除缓存，强制让这些失败的化合物重新去网络上查一遍，而不是读取刚才失败的记录
    f_fetch_compound_info.cache_clear()
    
    logging.info("=== 启动第二轮补漏检索 (Retry-on-Miss) ===")
    total_retry = len(failed_names)
    for idx, name in enumerate(failed_names, start=1):
        print(f"\n" + "="*60)
        print(f"🚑 补漏进度: [{idx}/{total_retry}] | 正在重新抢救: {name}")
        
        # 在重试前额外增加一个休眠，稳扎稳打
        time.sleep(2) 
        print(f"--> 正在再次向 PubChem 发起请求 ...", flush=True)
        
        retry_res = f_fetch_compound_info(name)
        
        # 覆盖刚才失败的记录
        results_dict[name] = retry_res
        
        if None not in retry_res:
            logging.info(f"🎉 抢救成功！已成功补齐 {name} 的所有字段。")
        else:
            logging.warning(f"❌ 抢救失败，{name} 的数据在 PubChem 中可能确实不存在或存在持续网络阻断。")
else:
    logging.info("\n🎉 第一轮检索非常完美！所有化合物数据均完整，无需重试。")

# ---------------- Phase 3: 整理与保存 ----------------
print(f"\n" + "="*60)
logging.info("所有处理完毕，正在整合并保存数据...")

# 按照原本 DataFrame 的顺序把结果抽出来
final_results = [results_dict[name] for name in compounds_table['compound']]

result_df = pd.DataFrame(
    final_results, 
    index=compounds_table.index, 
    columns=['ismiles', 'uname', 'cas']
)

ids_map_table = pd.concat([compounds_table, result_df], axis=1)

2026-03-11 10:09:28,600 - INFO - 开始加载数据集...
2026-03-11 10:09:28,657 - INFO - 提取到 311 个需要处理的唯一化合物。
2026-03-11 10:09:28,658 - INFO - === 启动第一轮全量检索 ===

📦 初次检索进度: [1/311] | 当前处理: Zinc chloride
🚀 总体进度: |----------------------------------------| 0.3%
--> 正在向 PubChem 请求 ...

📦 初次检索进度: [2/311] | 当前处理: 1-Butyl-3-methylimidazolium chloride
🚀 总体进度: |----------------------------------------| 0.6%
--> 正在向 PubChem 请求 ...

📦 初次检索进度: [3/311] | 当前处理: FeCl3
🚀 总体进度: |----------------------------------------| 1.0%
--> 正在向 PubChem 请求 ...

📦 初次检索进度: [4/311] | 当前处理: Tetrabutylphosphonium bromide
🚀 总体进度: |----------------------------------------| 1.3%
--> 正在向 PubChem 请求 ...

📦 初次检索进度: [5/311] | 当前处理: AlCl3
🚀 总体进度: |----------------------------------------| 1.6%
--> 正在向 PubChem 请求 ...

📦 初次检索进度: [6/311] | 当前处理: N-benzyl-N,N-dibutylbutan-1-aminium chloride
🚀 总体进度: |----------------------------------------| 1.9%
--> 正在向 PubChem 请求 ...

📦 初次检索进度: [7/311] | 当前处理: CuCl2
🚀 总体进度: |-----------------------------------

In [7]:
os.makedirs(os.path.dirname(Config.IDS_MAP_TABLE) or '.', exist_ok=True)
ids_map_table.to_csv(Config.IDS_MAP_TABLE, index=False)

os.makedirs(os.path.dirname(Config.IDS_MAP_TABLE_OUT) or '.', exist_ok=True)
ids_map_table.to_csv(Config.IDS_MAP_TABLE_OUT, index=False)

logging.info(f"✅ 处理完成！结果已成功保存至: {Config.IDS_MAP_TABLE} and {Config.IDS_MAP_TABLE_OUT}")

2026-03-11 10:37:13,944 - INFO - ✅ 处理完成！结果已成功保存至: ../data/in_out\260311_raw-ds_fetch-ids_add-ids.csv and ../data/output\260311_out_ids-map-tb.csv


人工检查数据集质量，填充空值

In [8]:
qlty_sum_output_path = os.path.join(Config.MISSING_DATA_MAG_DIR, 'ids-map-tb_qlty-sum.xlsx')
data_quality_summary(ids_map_table, qlty_sum_output_path)

In [9]:
ids_map_table = pd.read_csv(Config.IDS_MAP_TABLE)
ids_map_table = ids_map_table.dropna()

In [10]:
group_cols = ['ismiles', 'uname', 'cas']
byproduct_table = deduplicate_byproduct_table(ids_map_table, group_cols, Config.DEDUPLICATE_DIR )
byproduct_table.to_csv(Config.BYPRODUCT_TABLE , index=False)
byproduct_table.to_csv(Config.BYPRODUCT_TABLE_OUT , index=False)

总重复行数: 149
原始行数: 308
去重后行数: 221
移除重复行数: 87
所有重复记录行数: 149
被移除的重复行数: 87
